# Supervised FineTuning (SFT) con LoRa sobre un modelo pequeño Qwen

Posible código a usar:

[LoRA](https://kaifimtz.medium.com/fine-tuning-large-language-models-sft-lora-and-qlora-explained-d97bd4c1504e)


Instalemos las librerías necesarias a usar e instanciemos los datos desde Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q -U transformers peft datasets accelerate trl

Importemos las librerías a usar

In [3]:
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("Soporta BF16:", torch.cuda.is_bf16_supported())

GPU: Tesla T4
VRAM (GB): 15.6
Soporta BF16: True


In [4]:
import json, random
from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
import matplotlib.pyplot as plt

Definamos los parámetros del modelo

In [5]:
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SFT_PATH = "/content/drive/MyDrive/GenAI/sft.jsonl"
MAX_LEN = 3200  # Cubre system prompt (~1200 toks) + hasta 2 turnos SQL #TODO: por definir
OUT_DIR = "/content/drive/MyDrive/GenAI/e2/checkpoints/sft_lora"
USE_BF16 = torch.cuda.is_bf16_supported()
SEED = 42

Importamos el tokenizer de Qwen

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer

Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-Coder-1.5B-Instruct', vocab_size=151643, model_max_length=32768, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstr

Carguemos los datos

In [7]:
rows = [json.loads(line) for line in open(SFT_PATH, encoding='utf-8') if line.strip()]
rows[0], "#################################### uwu ##############################", rows[1:3]

({'task_id': 'hosp_0001',
  'messages': [{'role': 'system',
    'content': "Eres un agente experto en bases de datos SQLite para un entorno hospitalario.\nTu objetivo es responder con exactitud a la pregunta del usuario consultando la base de datos.\n\n### PROTOCOLO DE INTERACCIÓN POR TURNO:\n- Si necesitas consultar la base de datos (o corregir una consulta anterior): emite un bloque <sql>SELECT ...</sql> y detén tu respuesta inmediatamente para esperar el bloque <result>...</result> del sistema.\n- Si ya recibiste un <result>...</result> válido con los datos que responden la pregunta: emite un único bloque <answer>...</answer> con el dato final y termina.\n- Opcionalmente puedes razonar brevemente antes dentro de <think>...</think>.\n- PROHIBIDO incluir <sql> y <answer> en el mismo turno.\n- PROHIBIDO inventar bloques <result>...</result>.\n\n### REGLAS DE SQLITE Y CORRECCIÓN DE ERRORES:\n1. <sql>...</sql>:\n   - Escribe UNA ÚNICA sentencia SELECT válida para SQLite (solo lectura), s

In [8]:
task_ids = sorted({r["task_id"] for r in rows})
task_ids[0:3]

['hosp_0001', 'hosp_0002', 'hosp_0004']

In [9]:
random.Random(SEED).shuffle(task_ids)
task_ids[0:3]

['hosp_0750', 'hosp_1524', 'hosp_1885']

Usemos el 5% del dataset para validation. Por tanto, usaremos un 95% para train split

In [10]:
VAL_SPLIT_PCT = 0.05
TRAIN_SPLIT_PCT = 1 - VAL_SPLIT_PCT

val_ids = set(task_ids[: int(len(task_ids) * VAL_SPLIT_PCT)])
val_ids

{'hosp_0008',
 'hosp_0026',
 'hosp_0103',
 'hosp_0175',
 'hosp_0191',
 'hosp_0236',
 'hosp_0246',
 'hosp_0318',
 'hosp_0357',
 'hosp_0399',
 'hosp_0553',
 'hosp_0567',
 'hosp_0579',
 'hosp_0612',
 'hosp_0750',
 'hosp_0785',
 'hosp_0813',
 'hosp_0825',
 'hosp_0849',
 'hosp_0855',
 'hosp_0858',
 'hosp_0869',
 'hosp_1165',
 'hosp_1229',
 'hosp_1242',
 'hosp_1332',
 'hosp_1385',
 'hosp_1393',
 'hosp_1475',
 'hosp_1481',
 'hosp_1499',
 'hosp_1506',
 'hosp_1508',
 'hosp_1524',
 'hosp_1556',
 'hosp_1616',
 'hosp_1617',
 'hosp_1656',
 'hosp_1665',
 'hosp_1679',
 'hosp_1698',
 'hosp_1832',
 'hosp_1853',
 'hosp_1885',
 'hosp_1950',
 'hosp_1977',
 'hosp_1985',
 'hosp_2013',
 'hosp_2030',
 'hosp_2066',
 'hosp_2255',
 'hosp_2288',
 'hosp_2344',
 'hosp_2393',
 'hosp_2399',
 'hosp_2461',
 'hosp_2484'}

Obtenemos los datos crudos

In [11]:
train_raw = [r for r in rows if r["task_id"] not in val_ids]
train_raw[0]

{'task_id': 'hosp_0001',
 'messages': [{'role': 'system',
   'content': "Eres un agente experto en bases de datos SQLite para un entorno hospitalario.\nTu objetivo es responder con exactitud a la pregunta del usuario consultando la base de datos.\n\n### PROTOCOLO DE INTERACCIÓN POR TURNO:\n- Si necesitas consultar la base de datos (o corregir una consulta anterior): emite un bloque <sql>SELECT ...</sql> y detén tu respuesta inmediatamente para esperar el bloque <result>...</result> del sistema.\n- Si ya recibiste un <result>...</result> válido con los datos que responden la pregunta: emite un único bloque <answer>...</answer> con el dato final y termina.\n- Opcionalmente puedes razonar brevemente antes dentro de <think>...</think>.\n- PROHIBIDO incluir <sql> y <answer> en el mismo turno.\n- PROHIBIDO inventar bloques <result>...</result>.\n\n### REGLAS DE SQLITE Y CORRECCIÓN DE ERRORES:\n1. <sql>...</sql>:\n   - Escribe UNA ÚNICA sentencia SELECT válida para SQLite (solo lectura), sin 

In [12]:
val_raw = [r for r in rows if r["task_id"] in val_ids]
val_raw[0]

{'task_id': 'hosp_0008',
 'messages': [{'role': 'system',
   'content': "Eres un agente experto en bases de datos SQLite para un entorno hospitalario.\nTu objetivo es responder con exactitud a la pregunta del usuario consultando la base de datos.\n\n### PROTOCOLO DE INTERACCIÓN POR TURNO:\n- Si necesitas consultar la base de datos (o corregir una consulta anterior): emite un bloque <sql>SELECT ...</sql> y detén tu respuesta inmediatamente para esperar el bloque <result>...</result> del sistema.\n- Si ya recibiste un <result>...</result> válido con los datos que responden la pregunta: emite un único bloque <answer>...</answer> con el dato final y termina.\n- Opcionalmente puedes razonar brevemente antes dentro de <think>...</think>.\n- PROHIBIDO incluir <sql> y <answer> en el mismo turno.\n- PROHIBIDO inventar bloques <result>...</result>.\n\n### REGLAS DE SQLITE Y CORRECCIÓN DE ERRORES:\n1. <sql>...</sql>:\n   - Escribe UNA ÚNICA sentencia SELECT válida para SQLite (solo lectura), sin 

In [13]:
print(f"Train: {len(train_raw)} trazas | Val: {len(val_raw)} trazas")

Train: 2834 trazas | Val: 159 trazas


## Tokenización

Podriamos usa `apply_chat_template` de HuggingFace, que viene ya con el tokenizer. Pero no quize usarlo por ahora, aunque podría ser recomendable solo por mantener la consistencia. Pero habría que verificar que realmente es correcto el método. Además, igualmente habría que enmascarar con -100 las partes que no queremos que aprenda el modelo.

In [14]:
def tokenize_with_masking(example):
  input_ids = []
  labels_ids = []

  for msg in example["messages"]:
    role = msg["role"]
    content = msg["content"]

    # Agregamos el formato que necesita Qwen
    header_ids = tokenizer.encode(f"<|im_start|>{role}\n", add_special_tokens=False)
    body_ids   = tokenizer.encode(f"{content}<|im_end|>\n", add_special_tokens=False)

    input_ids.extend(header_ids + body_ids)

    if role == "assistant":
      # Enmascaramos encabezado "<|im_start|>assistant\n" solamente. Hay que calcular la
      # loss en la respuesta + <|im_end|>

      # Ponemos el id: -100 por cada uno de los header_ids. Luego, agregamos los
      # body_id tal cual
      labels_ids.extend([-100] * len(header_ids) + body_ids)

    else:
      # system y user (tools) roles, ponemos el id: -100 en todo, porque no
      # queremos que
      # se fije en esto. Solo que se concentr en predecir bien lo del asistente
      labels_ids.extend([-100] * (len(header_ids) + len(body_ids)))

  return {
      "input_ids": input_ids[:MAX_LEN],
      "labels": labels_ids[:MAX_LEN],
      "attention_mask": [1] * min(len(input_ids), MAX_LEN),
  }


In [15]:
train_ds = Dataset.from_list(train_raw).map(tokenize_with_masking, remove_columns=["task_id", "messages"])
val_ds   = Dataset.from_list(val_raw).map(tokenize_with_masking, remove_columns=["task_id", "messages"])

Map:   0%|          | 0/2834 [00:00<?, ? examples/s]

Map:   0%|          | 0/159 [00:00<?, ? examples/s]

## Verificación de tokenización correcta

In [16]:
sample = train_ds[0]
active_ids = [tid for tid, lbl in zip(sample["input_ids"], sample["labels"]) if lbl != -100]
print("=== TOKENS CON LOSS ACTIVA (solo debe aparecer <sql> y <answer>) ===")
print(tokenizer.decode(active_ids))
print("\nTokens totales:", len(sample["input_ids"]), "| Tokens con loss:", len(active_ids))
print("Largo máximo en train_ds:", max(len(x["input_ids"]) for x in train_ds))

=== TOKENS CON LOSS ACTIVA (solo debe aparecer <sql> y <answer>) ===
<sql>SELECT SUM(a.cost) FROM appointments a JOIN doctors d ON a.doctor_id = d.doctor_id JOIN departments dp ON d.department_id = dp.department_id WHERE dp.name = 'Medicina Interna' AND a.appointment_date LIKE '2024-11-%' AND a.status = 'completed'</sql><|im_end|>
<answer>282000</answer><|im_end|>


Tokens totales: 2467 | Tokens con loss: 86
Largo máximo en train_ds: 2560


## Instanciar el modelo LoRa

In [17]:
# Cargamos modelo base en BF16 (L4/A100) o FP16 (T4)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    device_map="auto",
)
model.config.use_cache = False

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## Inyectar adaptadores LoRA

In [18]:
# Inyectamos los adaptadores LoRA
# TODO: Investigar que hiperparámetros vamos a usar

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:195: UserWarning: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported. torchao support in PEFT is disabled; if you want to use a torchao-quantized model with PEFT, please upgrade torchao.
  warnings.warn(


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Se identifican trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

## Definición de Collator

In [19]:
# Collator que rellena con pad_token_id en input_ids y con -100 en labels
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)

## Entrenar y guardar adaptador LoRA

Definimos aqui en la misma celda los hiperparámetros

In [20]:
# # TODO: Ver WarmUp

# training_args = TrainingArguments(
#     output_dir=OUT_DIR,
#     num_train_epochs=3,
#     per_device_train_batch_size=1 if not USE_BF16 else 2,  # 1 en T4, 2 en L4
#     gradient_accumulation_steps=16 if not USE_BF16 else 8, # Batch efectivo = 16 en ambos
#     learning_rate=2e-4,
#     lr_scheduler_type="cosine",
#     weight_decay=0.01,
#     gradient_checkpointing=True,        # Clave para ahorrar ~50% VRAM
#     gradient_checkpointing_kwargs={"use_reentrant": False},
#     bf16=USE_BF16,
#     fp16=not USE_BF16,
#     logging_steps=10,
#     eval_strategy="steps",
#     eval_steps=50,
#     save_strategy="epoch",              # Respalda en Drive al final de cada época
#     save_total_limit=2,
#     report_to="none",
#     seed=2026,
# )
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_ds,
#     eval_dataset=val_ds,
#     data_collator=collator,
# )
# trainer.train()



In [22]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

training_args = TrainingArguments(
    per_device_eval_batch_size=1,   # Baja los logits de 11.4 GiB -> 1.4 GiB (¡8x menos!)
    prediction_loss_only=True,      # Solo calcula el número eval_loss; NO acumula los logits gigantes en GPU
    output_dir=OUT_DIR,
    max_steps=5,                      # <-- Solo 5 pasos para probar
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,    # <-- 4x más rápido por paso solo para testear
    logging_steps=1,                  # <-- Imprime la loss en CADA paso (Step 1, 2, 3...)
    eval_strategy="steps",
    eval_steps=5,                     # <-- Prueba también que eval_loss corre sin OOM
    save_strategy="no",               # <-- No ocupa espacio en Drive durante el test
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=USE_BF16,
    fp16=not USE_BF16,
    learning_rate=2e-4,
    report_to="none",
    seed=2026,
)

# Usa solo 10 ejemplos de validación para que eval_steps=5 termine en segundos:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds.select(range(10)),
    data_collator=collator,
)
trainer.train()

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing_kwargs` is deprecated and will be removed in v6. Use `activation_checkpointing_kwargs` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Step,Training Loss,Validation Loss
5,0.076539,0.054481


TrainOutput(global_step=5, training_loss=0.07009050101041794, metrics={'train_runtime': 312.8992, 'train_samples_per_second': 0.064, 'train_steps_per_second': 0.016, 'total_flos': 394113044078592.0, 'train_loss': 0.07009050101041794, 'epoch': 0.007057163020465773})

In [ ]:
## Guardar adaptador final
trainer.save_model(f"{OUT_DIR}/final")
tokenizer.save_pretrained(f"{OUT_DIR}/final")

## Evaluación

Graficamos la pérdida obtenida

In [ ]:
logs = trainer.state.log_history
train_steps = [x["step"] for x in logs if "loss" in x]
train_loss  = [x["loss"] for x in logs if "loss" in x]
eval_steps  = [x["step"] for x in logs if "eval_loss" in x]
eval_loss   = [x["eval_loss"] for x in logs if "eval_loss" in x]

plt.figure(figsize=(8, 4))
plt.plot(train_steps, train_loss, label="Train Loss (Assistant-Only)")
plt.plot(eval_steps, eval_loss, marker="o", label="Val Loss (5% tareas)")
plt.xlabel("Step")
plt.ylabel("Cross-Entropy Loss")
plt.title("Curva SFT LoRA — Qwen2.5-Coder-1.5B-Instruct")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(f"{OUT_DIR}/sft_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()